#### Import Libraries

In [1]:
import cv2 as cv
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

#### Analysis With Images
Collect Images, Capture Faces, Save Faces For Training

In [2]:
# Detect the face from image and than crop image only to face
haar_cascade = cv.CascadeClassifier("./haar_face.xml")

In [3]:
# Look at images
path = r"samples"
imgs = os.listdir(path)
imgs

['$RBATYQN.jpg',
 '$RX5JFK4.jpg',
 '1.jpg',
 '20201105_162309.jpg',
 'copy IMG_20221002_070653723.jpg',
 'IMG-20210209-WA0011.jpg',
 'IMG-20210209-WA0015.jpg',
 'IMG-20210209-WA0015.png',
 'IMG-20210306-WA0029.jpg',
 'IMG-20210306-WA0029.png',
 'IMG-20210721-WA0044.jpg',
 'IMG-20210722-WA0011.jpg',
 'IMG-20210722-WA0015.jpg',
 'IMG_-45zj6a.jpg',
 'IMG_0004.jpg',
 'IMG_20211207_102822.jpg',
 'IMG_20221215_001747.jpg',
 'IMG_3327.png',
 'IMG_3328.png',
 'IMG_3331.png',
 'IMG_3332.png',
 'IMG_3333.png',
 'IMG_3334.png',
 'IMG_9882.jpg',
 'Snapchat-1122651459.jpg',
 'Snapchat-1593342754.png',
 'Snapchat-29508185.jpg',
 'Snapchat-366152141.jpg',
 'Snapchat-566259894.jpg',
 'Snapchat-812400195.jpg']

In [5]:
# Crop Faces and save them into train folder

for img in imgs:
    imgFile = os.path.join(path,img)
    imgRead = cv.imread(imgFile)

    # half_width = int(imgRead.shape[0]/2)
    # half_heght = int(imgRead.shape[1]/2)
    
    # imgRead = cv.resize(imgRead,(half_width,half_heght))

    face_detect = haar_cascade.detectMultiScale(imgRead,scaleFactor=1.8,minNeighbors=1)
    print(f"Faces Detected: {len(face_detect)}")

    for (x,y,w,h) in face_detect:
        face = imgRead[y:h+y,x:x+w]
        cv.imwrite(f"./train/{img}",face)
cv.waitKey(0)

Faces Detected: 1
Faces Detected: 0
Faces Detected: 1
Faces Detected: 2
Faces Detected: 1
Faces Detected: 1
Faces Detected: 1
Faces Detected: 1
Faces Detected: 1
Faces Detected: 1
Faces Detected: 1
Faces Detected: 1
Faces Detected: 1
Faces Detected: 2
Faces Detected: 2
Faces Detected: 1
Faces Detected: 3
Faces Detected: 2
Faces Detected: 2
Faces Detected: 2
Faces Detected: 2
Faces Detected: 1
Faces Detected: 1
Faces Detected: 9
Faces Detected: 1
Faces Detected: 1
Faces Detected: 1
Faces Detected: 0
Faces Detected: 1
Faces Detected: 1


-1

In [3]:
# Look at croped images
path = r"train"
peoples = os.listdir(path)
peoples

['$RBATYQN.jpg',
 '1.jpg',
 'copy IMG_20221002_070653723.jpg',
 'IMG-20210209-WA0011.jpg',
 'IMG-20210209-WA0015.jpg',
 'IMG-20210209-WA0015.png',
 'IMG-20210306-WA0029.jpg',
 'IMG-20210306-WA0029.png',
 'IMG-20210721-WA0044.jpg',
 'IMG-20210722-WA0011.jpg',
 'IMG-20210722-WA0015.jpg',
 'IMG_0004.jpg',
 'IMG_20211207_102822.jpg',
 'IMG_3331.png',
 'IMG_3333.png',
 'IMG_3334.png',
 'Snapchat-1122651459.jpg',
 'Snapchat-1593342754.png',
 'Snapchat-29508185.jpg',
 'Snapchat-566259894.jpg',
 'Snapchat-812400195.jpg']

In [4]:
features = []
labels = []

In [5]:
# Extract Images as Features
for person in peoples:
    label = peoples.index(person)
    img_array = cv.imread(f"./train/{person}")
    put_image = cv.cvtColor(img_array, cv.COLOR_RGB2GRAY)
    features.append(put_image)
    labels.append(label)

In [6]:
# Convert to Array of objects
features = np.array(features ,dtype='object')
labels = np.array(labels)

In [7]:
# Create Face Recognizer
face_recognizer = cv.face.LBPHFaceRecognizer.create()
face_recognizer.train(features,labels)
face_recognizer.save("face_trained.yml")

In [8]:
# Save features and Labels
np.save("features.npy",features)
np.save("labels.npy",labels)

In [9]:
# Test
face_recognizer.read("face_trained.yml")

In [10]:
# Detect Through Image
frame = cv.imread("./test/1.jpg")
gray = cv.cvtColor(frame,cv.COLOR_BGR2GRAY)
cv.imshow("Detecting ...",gray)
# Detect the face in image
face_rect = haar_cascade.detectMultiScale(gray,scaleFactor=1.8,minNeighbors=1)
for (x,y,w,h) in face_rect:
    faces_roi = gray[y:y+h,x:x+w]
    label, confidence = face_recognizer.predict(faces_roi)
    print((confidence))
    if(np.ceil(confidence) > 0):
        print("Sakhawat ...")
        cv.putText(frame,"Sakhawat",(10,50),cv.FONT_HERSHEY_DUPLEX,3,(255,0,0),3)
        cv.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),thickness=3)
    else:
        print("Unknown")
        cv.putText(frame,"Unknown",(10,50),cv.FONT_HERSHEY_DUPLEX,3,(255,0,0),3)
        cv.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),thickness=3)
cv.imshow("Detected Face ...",frame)
cv.waitKey(0)

-1

In [11]:
# Live From Video
capture = cv.VideoCapture(0)
counter = 0
while True:
    ret, frame = capture.read()
    if(ret):
        counter = counter+1
        if(counter % 30 == 0):
            gray = cv.cvtColor(frame,cv.COLOR_BGR2GRAY)
            cv.imshow("Detecting ...",gray)
            # Detect the face in image
            face_rect = haar_cascade.detectMultiScale(gray,scaleFactor=1.8,minNeighbors=1)
            for (x,y,w,h) in face_rect:
                faces_roi = gray[y:y+h,x:x+w]
                label, confidence = face_recognizer.predict(faces_roi)
                # print((confidence))
                if(np.ceil(confidence) > 0):
                    print("Sakhawat ...")
                    cv.putText(frame,"Sakhawat",(10,50),cv.FONT_HERSHEY_DUPLEX,3,(255,0,0),3)
                    cv.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),thickness=3)
                else:
                    print("Unknown")
                    cv.putText(frame,"Unknown",(10,50),cv.FONT_HERSHEY_DUPLEX,3,(255,0,0),3)
                    cv.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),thickness=3)
    else:
        break

cv.imshow("Detected Face ...",frame)
cv.waitKey(0)
cv.destroyAllWindows()

Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...
Sakhawat ...


: 